# CrisisWorldCortex — Colab Mini-Demo (Unsloth + SmolLM2-360M)

**Purpose.** Stand-alone Colab notebook that demonstrates the GRPO training pipeline end-to-end on free Colab resources. Mirrors the production pipeline in `training/scripts/train_b1_grpo.py` but with:

- **Model:** `unsloth/SmolLM2-360M-Instruct` (~360M params, well under the 500M ceiling for free Colab).
- **Steps:** `MAX_TRAIN_STEPS = 20` (vs. 300 in production). Just enough to show learning signal flowing.
- **Dataset:** 2 tasks × 5 seeds = 10 prompts (vs. 3 tasks × 50 seeds = 150 in production).
- **Group size:** 2 (vs. 4 in production).

Wall-clock: ~5–10 minutes on a free Colab T4. CPU-only fallback also works (slower).

**Compatibility note.** The user's production HF Jobs runs use `train_b1_grpo.py` with Qwen3-7B / Llama-3.1-8B. This notebook does **not** replace those; it is a demo-grade smoke run for showing the loop turning. The training contract — single-step GRPO against the Phase-1-fixed `outer_reward`, parse-failure → marker action — is identical.

**Prereqs.**
1. Colab Secrets has `HF_TOKEN` set (Tools → Secrets, name = `HF_TOKEN`, value = `hf_xxx` with write access). Skip if you don't want to push to Hub.
2. The HF Space `Angshuman28/CrisisWorldCortex` is running and reachable. (You can override `ENV_URL` below if you've forked it.)

**One-shot run:** `Runtime → Run all`. The notebook prints a clear status line at the end indicating whether the loop completed, and prints reward statistics so you can see signal.

## 1. Install dependencies

Mirrors the install cell in `notebooks/train_b1_grpo.ipynb`. Unsloth ships its own torch / vLLM / xformers stack tuned for Colab T4.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-deps "trl>=0.14" peft accelerate bitsandbytes
!pip install "openenv-core[core]==0.2.1" "fastmcp<3.0.0" "pydantic>=2.10.6,<2.11" mergekit huggingface_hub matplotlib datasets

## 2. Authenticate with Hugging Face (optional)

Reads `HF_TOKEN` from Colab Secrets. If unavailable, the notebook still runs end-to-end — the final Hub push step will be skipped.

In [ ]:
import os

HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF auth OK — Hub push enabled at the end.")
else:
    print("No HF_TOKEN found. Notebook will still run; Hub push step will be skipped.")

## 3. Clone CrisisWorldCortex and install

Pulls the deployed HF Space repo and installs it locally. This gives the notebook the same `CrisisworldcortexEnv` HTTP client, `baselines.flat_agent` system prompt, and parser used by the production pipeline. **Nothing in the repo is modified — this is a read-only consumer.**

In [ ]:
%%capture
!rm -rf /content/CrisisWorldCortex
!git clone https://huggingface.co/spaces/Angshuman28/CrisisWorldCortex /content/CrisisWorldCortex
%cd /content/CrisisWorldCortex
!pip install -e . --no-deps

In [ ]:
import sys
sys.path.insert(0, "/content/CrisisWorldCortex")

from baselines.flat_agent import (
    build_system_prompt,
    parse_action,
    parse_failure_marker,
    serialize_observation,
)
from CrisisWorldCortex import CrisisworldcortexAction, CrisisworldcortexObservation
from CrisisWorldCortex.client import CrisisworldcortexEnv

print("CrisisWorld imports OK")

## 4. Load SmolLM2-360M with Unsloth + LoRA

SmolLM2-360M is ~360M parameters — well under the user-requested 500M ceiling and comfortable on free Colab T4 (or CPU as fallback). LoRA rank 16 is plenty for a smoke demo.

Alternative tiny models, all Unsloth-supported, all under 500M:
- `unsloth/SmolLM2-135M-Instruct` (135M — fastest)
- `unsloth/SmolLM2-360M-Instruct` (360M — current default, best signal-to-cost)
- `unsloth/Qwen2.5-0.5B-Instruct` (~500M — closest in family to the production Qwen3 baseline)

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/SmolLM2-360M-Instruct"
MAX_SEQ_LEN = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,  # vLLM-backed generate, required by GRPOTrainer
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Model + LoRA loaded — {MODEL_NAME}")

## 5. Connect to the deployed CrisisWorld env

Same HTTP client + URL as the production notebook. Each rollout = one `env.reset()` + one `env.step()` (single-step GRPO; matches the binding contract in `training/CLAUDE.md`).

In [ ]:
ENV_URL = "https://angshuman28-crisisworldcortex.hf.space"
TASKS = ("outbreak_easy", "outbreak_medium")  # 2 tasks for the mini-demo
EPISODE_TICKS = 12
SEEDS_PER_TASK = 5


def _sync_if_available(env):
    # OpenEnv 0.2.2+ exposes .sync(); OpenEnv 0.2.1 reset/step are already sync.
    sync = getattr(env, "sync", None)
    return sync() if callable(sync) else env


def make_env():
    return _sync_if_available(CrisisworldcortexEnv(base_url=ENV_URL))


# Sanity: one full reset round-trip. Always close the env afterwards —
# the deployed HF Space drops idle websockets and re-using a stale client
# raises ConnectionClosedOK on the next call.
_test_env = make_env()
try:
    _result = _test_env.reset(task_name="outbreak_easy", seed=0, max_ticks=EPISODE_TICKS)
    _obs = _result.observation if hasattr(_result, "observation") else _result
    print(f"Env OK. Initial tick={_obs.tick}, regions={[r.region for r in _obs.regions]}")
finally:
    _test_env.close()

## 6. Build the prompt dataset

10 prompts total (2 tasks × 5 seeds). Production uses 150.

In [ ]:
import random
from datasets import Dataset

SYSTEM_PROMPT = build_system_prompt()


def make_chat_prompt(obs: CrisisworldcortexObservation) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            # last_reward=0.0 — fresh reset, no prior step.
            {"role": "user", "content": serialize_observation(obs, last_reward=0.0)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


rng = random.Random(0)
_seed_pool = [
    {"task": task, "seed": seed}
    for task in TASKS
    for seed in range(SEEDS_PER_TASK)
]
rng.shuffle(_seed_pool)

_prompts: list[str] = []
_meta: list[dict] = []
for entry in _seed_pool:
    env = make_env()
    try:
        result = env.reset(task_name=entry["task"], seed=entry["seed"], max_ticks=EPISODE_TICKS)
        obs = result.observation if hasattr(result, "observation") else result
        _prompts.append(make_chat_prompt(obs))
        _meta.append(entry)
    finally:
        env.close()

train_dataset = Dataset.from_dict(
    {
        "prompt": _prompts,
        "task": [m["task"] for m in _meta],
        "seed": [m["seed"] for m in _meta],
    }
)
print(f"Dataset built: {len(train_dataset)} examples")

## 7. Reward function

Identical contract to the production trainer: parse the completion → submit one env step → return `obs.reward` ∈ `[-1.0, 1.0]`. Parse failures hit the §19 marker-action path (`-1.0` + terminate).

In [ ]:
def crisisworld_reward(
    prompts: list[str],
    completions: list[str],
    task: list[str],
    seed: list[int],
    **_kwargs: object,
) -> list[float]:
    rewards: list[float] = []
    for completion, t, s in zip(completions, task, seed):
        env = make_env()
        try:
            env.reset(task_name=t, seed=int(s), max_ticks=EPISODE_TICKS)
            action_payload = parse_action(completion) or parse_failure_marker()
            try:
                result = env.step(CrisisworldcortexAction(action=action_payload))
                obs = result.observation if hasattr(result, "observation") else result
                reward = obs.reward if obs.reward is not None else 0.0
                rewards.append(float(reward))
            except Exception as exc:
                print(f"[WARN] env.step failed task={t} seed={s}: {exc}")
                rewards.append(-1.0)
        finally:
            env.close()
    return rewards

## 8. GRPO training (mini)

20 steps × group size 2 = 40 rollouts. Wall-clock ~5–10 minutes on T4. The point of this notebook is to **show the loop running and producing rewards**, not to converge.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MAX_TRAIN_STEPS = 20
GROUP_SIZE = 2
MAX_PROMPT_LEN = 1536
MAX_COMPLETION_LEN = 256

training_args = GRPOConfig(
    output_dir="/content/colab_demo_lora",
    learning_rate=5e-6,
    per_device_train_batch_size=GROUP_SIZE,
    gradient_accumulation_steps=1,
    num_generations=GROUP_SIZE,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_TRAIN_STEPS,
    save_steps=MAX_TRAIN_STEPS,  # save once at the end
    logging_steps=1,
    report_to="none",
    bf16=True,
    optim="adamw_8bit",
    temperature=0.8,
    use_vllm=True,
    vllm_mode="colocate",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[crisisworld_reward],
    args=training_args,
    train_dataset=train_dataset,
)
print("GRPOTrainer constructed; starting train()...")

In [ ]:
trainer.train()
print("\n=== Training finished ===")

## 9. Reward summary

Pulls the per-step rewards from the trainer log and plots them. With only 20 steps you should not expect convergence — the goal is to confirm rewards are non-trivially distributed and the gradient pipeline is alive.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
reward_steps = [
    (entry.get("step", i), entry["reward"])
    for i, entry in enumerate(log_history)
    if "reward" in entry
]

if reward_steps:
    steps, rewards = zip(*reward_steps)
    print(f"Logged {len(rewards)} reward points.")
    print(f"  mean={sum(rewards)/len(rewards):.3f}  min={min(rewards):.3f}  max={max(rewards):.3f}")

    plt.figure(figsize=(8, 4))
    plt.plot(steps, rewards, marker="o", linewidth=1)
    plt.xlabel("GRPO step")
    plt.ylabel("Mean group reward")
    plt.title("CrisisWorldCortex mini-demo (SmolLM2-360M)")
    plt.axhline(0.0, linestyle=":", color="grey")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No 'reward' entries in log_history — check trainer logging_steps.")

## 10. (Optional) Push LoRA adapter to HF Hub

Skipped if no `HF_TOKEN` was found in step 2. Change `HUB_REPO` to your namespace if you want to push.

In [ ]:
OUTPUT_DIR = "/content/colab_demo_lora"
HUB_REPO = "Angshuman28/crisisworld-colab-demo-smollm2-360m"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved locally to {OUTPUT_DIR}")

if HF_TOKEN:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(
        HUB_REPO, exist_ok=True, repo_type="model", private=False, token=HF_TOKEN
    )
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_REPO,
        repo_type="model",
        token=HF_TOKEN,
    )
    print(f"Pushed to https://huggingface.co/{HUB_REPO}")
else:
    print("HF_TOKEN unset — skipping Hub push. Adapter is still saved locally.")

## 11. Quick eval — one episode per task

Runs a single 12-tick episode with the trained adapter on each task at `seed=0` and reports cumulative reward. With only 20 training steps this is a sanity check that the trained model still emits parseable actions, not an evaluation of policy quality.

In [ ]:
FastLanguageModel.for_inference(model)


def _hf_chat(system: str, user: str, max_new_tokens: int = 192) -> str:
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )


def run_one_episode(task: str, seed: int) -> tuple[float, int, int]:
    env = make_env()
    try:
        result = env.reset(task_name=task, seed=seed, max_ticks=EPISODE_TICKS)
        obs = result.observation if hasattr(result, "observation") else result
        cumulative = 0.0
        last_reward = 0.0
        parsed = 0
        failed = 0
        for _ in range(EPISODE_TICKS):
            completion = _hf_chat(SYSTEM_PROMPT, serialize_observation(obs, last_reward=last_reward))
            action = parse_action(completion)
            if action is None:
                failed += 1
                action = parse_failure_marker()
            else:
                parsed += 1
            result = env.step(CrisisworldcortexAction(action=action))
            obs = result.observation if hasattr(result, "observation") else result
            last_reward = obs.reward if obs.reward is not None else 0.0
            cumulative += last_reward
            if obs.done:
                break
        return cumulative, parsed, failed
    finally:
        env.close()


for task in TASKS:
    cum, parsed, failed = run_one_episode(task, seed=0)
    print(
        f"task={task:<18} cum_reward={cum:+.3f}  parsed={parsed:2d}  parse_failures={failed:2d}"
    )

print("\nMini-demo complete.")